<a href="https://colab.research.google.com/github/c81e32cat/kunteynir/blob/deepmultilingualpunctuation/kunteynir_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install pandas

Тексты, выгруженные с genius (при помощи `lyricsgenius` - библиотека не работает из google colab), приводим в нормальный вид (убираем аннотации, ссылки на обложки и т. д.)

In [12]:
import pandas as pd
import json
import re

with open('/content/Lyrics_Kunteynir_Fixed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

clean_data = []
for song in data['songs']:
    clean_data.append({
        'title': song['title'],
        'lyrics': song['lyrics']
    })

for i in range(len(clean_data)):
    curr_text=clean_data[i]['lyrics'].split('\n')
    new_text = []
    for string in curr_text:
        if string and string[0]!='—' and string not in new_text:
            new_text.append(string)
    new_text = '\n'.join(new_text)
    new_text = re.sub(r'\(.*?\)', '', new_text)
    clean_data[i]['lyrics']=new_text

df = pd.DataFrame(clean_data)

print(df.head())

                                  title  \
0        В аквапарке (At the Aqua Park)   
1                 Это жизнь (It’s Life)   
2              Чёрная дыра (Black Hole)   
3                     Клитор (Clitoris)   
4  Не прут колёса (Pills Doesn’t Catch)   

                                              lyrics  
0  В аквапарке реально охуенно, реально охуенно\n...  
1  Куча пенисов упирается мне в лицо\nЯ не вижу г...  
2  Вспоминаю случай с НЛО\nВозможно, чёрная дыра\...  
3  Не молчи больше так\nТолько не—\nЧтобы пригото...  
4  Выебут и даже не спросят\nНе прут колёса\nНе с...  


In [13]:
!pip install deepmultilingualpunctuation
!pip install transformers==4.30.2

  Using cached transformers-4.30.2-py3-none-any.whl.metadata (113 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.13.3.tar.gz (314 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached transformers-4.30.2-py3-none-any.whl (7.2 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [14]:
import sys
from transformers import pipeline
import deepmultilingualpunctuation

#возникли проблемы с transformers, так что gemini помог мне запустить библиотеку
def fixed_init(self, model='oliverguhr/fullstop-punctuation-multilang-large', device=None):
    import torch
    if device is None:
        device = 0 if torch.cuda.is_available() else -1

    self.pipe = pipeline('ner', model=model, device=device)

deepmultilingualpunctuation.PunctuationModel.__init__ = fixed_init

from deepmultilingualpunctuation import PunctuationModel
model = PunctuationModel()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: oliverguhr/fullstop-punctuation-multilang-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
dmp_data=[]

for i in range(len(clean_data)):
  text = clean_data[i]['lyrics'].replace('\n', ', ').lower()
  result = model.restore_punctuation(text)
  dmp_data.append({
        'title': clean_data[i]['title'],
        'lyrics': result
    })
  #print(dmp_data[i]['lyrics'].replace('.', '.\n'))

в аквапарке.
 реально охуенно, реально охуенно, ну реально правда охуенно в аквапарке.
 горки, джакузи, ванны и фонтан.
 с детьми были там вчера.
 разговариваю сам с собой, пуская слезу возле зеркала.
 очень хочу намутить себе.
 мерс намутил пиздатую проститутку, а она упорота анал использован славой ещё в ту пору, слил ей на куртку, размазал суши вокруг рта, колу обвафлила, сделала сука, напасов семнадцать и наблевала это.
 химическая реакция организма у аллы.
 забери свои потники, трогай эти лакосты, а то я въебу с локтя.
 тебе просто в клубе.
 охрана не успела разнять это как фисташки раскладывает мерчендайзер.
 насрал тебе на эквалайзер это как в пазике на суд отнеслась небрежно, как к ненужной посуде.
 привез посуду из турции.
 ты села жопой на тарелку.
 по сути меня посетили милые бляди, мило сосали на моих глазах, я в них спускал тогда защита: пизда на засов, прям еби в зад.
 я пиздатый в руке.
 strike.
 надо идти в аквапарк.
 мне похуй на растабайк, растаманку трахал кальмар ем

KeyboardInterrupt: 